<a href="https://colab.research.google.com/github/pcmouadji-dot/machine-learning/blob/main/House_Prices_Advanced_Regression_Techniques_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import sklearn as skl
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

# **importing and proccecing the trainning data.**

In [2]:
train_data=pd.read_csv("train.csv")
train_data['Alley']=train_data['Alley'].fillna('no_access')
train_data['LotArea']=np.log1p(train_data['LotArea'])#to deal withe the weilles.
train_data['LotFrontage']=train_data['LotFrontage'].fillna(train_data['LotFrontage'].median())
train_data=pd.get_dummies(train_data,columns=['Street'],dtype=int,prefix='Street')
train_data=pd.get_dummies(train_data,columns=['MSZoning'],dtype=int,prefix='MSZoning')
train_data=pd.get_dummies(train_data,columns=['Alley'],dtype=int,prefix='Alley')
train_data=pd.get_dummies(train_data,columns=['LandContour'],dtype=int,prefix='Contour')
dic={           20:	'1-STORY 1946 & NEWER ALL STYLES',
        30:	'1-STORY 1945 & OLDER',
        40:	'1-STORY W/FINISHED ATTIC ALL AGES',
        45:	'1-1/2 STORY - UNFINISHED ALL AGES',
        50:	'1-1/2 STORY FINISHED ALL AGES',
        60:	'2-STORY 1946 & NEWER',
        70:	'2-STORY 1945 & OLDER',
        75:	'2-1/2 STORY ALL AGES',
        80:	'SPLIT OR MULTI-LEVEL',
        85:	'SPLIT FOYER',
        90:	'DUPLEX - ALL STYLES AND AGES',
       120:	'1-STORY PUD (lanned Unit Development) - 1946 & NEWER',
       150:	'1-1/2 STORY PUD - ALL AGES',
       160:	'2-STORY PUD - 1946 & NEWER',
       180:	'PUD - MULTILEVEL - INCL SPLIT LEV/FOYER',
       190	:'2 FAMILY CONVERSION - ALL STYLES AND AGES'}
train_data['MSSubClass']=train_data['MSSubClass'].map(dic)
train_data=pd.get_dummies(train_data,columns=['MSSubClass'],dtype=int,prefix='MSSubClass')
con=(set(train_data['Condition1']) | set(train_data['Condition2']))
for c in con:
  if c!='Norm' :
    train_data[f'Condition{c}']=((train_data['Condition1']==c) | (train_data['Condition2']==c)).astype(int)
train_data=train_data.drop(columns=['Condition1','Condition2',])
train_data=pd.get_dummies(train_data,columns=['LotShape'],dtype=int,prefix='LotShape')
train_data=pd.get_dummies(train_data,columns=['BldgType'],dtype=int,prefix='BldgType')
train_data['Bathnum']=train_data['FullBath']+train_data['HalfBath']*0.5+train_data['BsmtHalfBath']*0.5+train_data['BsmtFullBath']
train_data=train_data.drop(columns=['BsmtFullBath','FullBath','HalfBath','BsmtHalfBath'])
train_data["age"]=(train_data['YrSold']-train_data['YearBuilt'])
train_data['IsRemodeled'] = (train_data['YearBuilt'] != train_data['YearRemodAdd']).astype(int)
train_data['MoSold']=train_data['MoSold'].astype(str)
train_data=pd.get_dummies(train_data,columns=['MoSold'],dtype=int,prefix='MoSold')
train_data['ismodeled']=(train_data['YearRemodAdd'] != train_data['YearBuilt']).astype(int)

train_data['TotalSF'] = train_data['1stFlrSF'] + train_data['2ndFlrSF'] + train_data['TotalBsmtSF']
train_data=train_data.drop(columns=['1stFlrSF','2ndFlrSF','TotalBsmtSF'])
train_data = pd.get_dummies(train_data, columns=['Neighborhood'], dtype=int, prefix='Neighborhood')
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'no_access': 0, 'None': 0}
qualities=['PoolQC','GarageCond','GarageQual','FireplaceQu','KitchenQual','HeatingQC','BsmtCond','BsmtQual','ExterCond','ExterQual']
for col in qualities:
  train_data[col]=train_data[col].fillna('None').map(qual_map)
util={'AllPub':4,'NoSewr':3,'NoSeWa':2,'ELO':1}
train_data['Utilities']=train_data['Utilities'].map(util)
train_data = pd.get_dummies(train_data, columns=['LotConfig'], dtype=int, prefix='Lot')
train_data['LandSlope']=train_data['LandSlope'].map({'Gtl':1,'Mod':2,'Sev':3})
train_data = pd.get_dummies(train_data, columns=['HouseStyle'], dtype=int, prefix='HouseStyle')
train_data = pd.get_dummies(train_data, columns=['RoofMatl'], dtype=int, prefix='RoofMatl')
train_data = pd.get_dummies(train_data, columns=['Foundation'], dtype=int, prefix='Foundation')
train_data = pd.get_dummies(train_data, columns=['RoofStyle'], dtype=int, prefix='RoofStyle')
train_data = pd.get_dummies(train_data, columns=['Heating'], dtype=int, prefix='Heating')
train_data = pd.get_dummies(train_data, columns=['Functional'], dtype=int, prefix='Functional')

con=(set(train_data['Exterior1st']) | set(train_data['Exterior2nd']))
for c in con:
  if c!='Norm' :
    train_data[f'Exterior{c}']=((train_data['Exterior1st']==c) | (train_data['Exterior2nd']==c)).astype(int)
train_data['MiscFeature']=train_data['MiscFeature'].fillna('other')
train_data = pd.get_dummies(train_data, columns=['MiscFeature'], dtype=int, prefix='MiscFeature')
train_data = pd.get_dummies(train_data, columns=['YrSold'], dtype=int, prefix='YrSold')
train_data = pd.get_dummies(train_data, columns=['SaleType', 'SaleCondition'], dtype=int)
bsmt_fin_map = {
    'GLQ': 6, 'ALQ': 5, 'BLQ': 4,
    'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0
}

for col in ['BsmtFinType1', 'BsmtFinType2']:
    train_data[col] = train_data[col].fillna('None').map(bsmt_fin_map)
exp_map = {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}
train_data['BsmtExposure'] = train_data['BsmtExposure'].fillna('None').map(exp_map)
gar_map = {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0}
train_data['GarageFinish'] = train_data['GarageFinish'].fillna('None').map(gar_map)
train_data['MasVnrArea'] = train_data['MasVnrArea'].fillna(0)
train_data['Electrical'] = train_data['Electrical'].fillna(train_data['Electrical'].mode()[0])
train_data = pd.get_dummies(train_data, columns=['Electrical', 'MasVnrType', 'GarageType'], dtype=int)
train_data = train_data.drop(columns=['Exterior1st', 'Exterior2nd', 'Id'])
train_data['SalePrice'] = np.log1p(train_data['SalePrice'])
bsmt_cols = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF']
for col in bsmt_cols:
    train_data[col] = train_data[col].fillna(0)
train_data['GarageYrBlt'] = train_data['GarageYrBlt'].fillna(train_data['YearBuilt'])
train_data['GarageCars'] = train_data['GarageCars'].fillna(0)
train_data['GarageArea'] = train_data['GarageArea'].fillna(0)
train_data['Fence']=train_data['Fence'].fillna('no').map({
           'GdPrv':4,
       'MnPrv'	: 3,
       'GdWo'	:2,
       'MnWw'	 :1,
       'no'	:0
})
train_data['PavedDrive']=train_data['PavedDrive'].map({
       'Y'	: 3,
       'P'	:2,
       'N'	 :1,

})
train_data['CentralAir']=train_data['CentralAir'].map({
       'Y'	: 1,
       'N'	 :0

})
print(train_data.select_dtypes(include=['object']).columns)
x=train_data.drop(columns='SalePrice')
y=train_data['SalePrice']
x.shape

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
scal=StandardScaler()
x_scal=scal.fit_transform(x)

model=LassoCV(alphas=[0.1,0.01,0.001,0.0005,0.000005],cv=5)
model.fit(x_scal,y)

In [ ]:
test_df_raw=pd.read_csv('test.csv')
id=test_df_raw['Id']
test_df_raw=test_df_raw.drop(columns=['Id'])

In [ ]:
test_data_processed=test_df_raw.copy()
test_data_processed['Alley']=test_data_processed['Alley'].fillna('no_access')
test_data_processed['LotArea']=np.log1p(test_data_processed['LotArea'])#to deal withe the weilles.
test_data_processed['LotFrontage']=test_data_processed['LotFrontage'].fillna(test_data_processed['LotFrontage'].median())
test_data_processed=pd.get_dummies(test_data_processed,columns=['Street'],dtype=int,prefix='Street')
test_data_processed=pd.get_dummies(test_data_processed,columns=['MSZoning'],dtype=int,prefix='MSZoning')
test_data_processed=pd.get_dummies(test_data_processed,columns=['Alley'],dtype=int,prefix='Alley')
test_data_processed=pd.get_dummies(test_data_processed,columns=['LandContour'],dtype=int,prefix='Contour')
dic={           20:	'1-STORY 1946 & NEWER ALL STYLES',
        30:	'1-STORY 1945 & OLDER',
        40:	'1-STORY W/FINISHED ATTIC ALL AGES',
        45:	'1-1/2 STORY - UNFINISHED ALL AGES',
        50:	'1-1/2 STORY FINISHED ALL AGES',
        60:	'2-STORY 1946 & NEWER',
        70:	'2-STORY 1945 & OLDER',
        75:	'2-1/2 STORY ALL AGES',
        80:	'SPLIT OR MULTI-LEVEL',
        85:	'SPLIT FOYER',
        90:	'DUPLEX - ALL STYLES AND AGES',
       120:	'1-STORY PUD (lanned Unit Development) - 1946 & NEWER',
       150:	'1-1/2 STORY PUD - ALL AGES',
       160:	'2-STORY PUD - 1946 & NEWER',
       180:	'PUD - MULTILEVEL - INCL SPLIT LEV/FOYER',
       190	:'2 FAMILY CONVERSION - ALL STYLES AND AGES'}
test_data_processed['MSSubClass']=test_data_processed['MSSubClass'].map(dic).fillna(0)
test_data_processed=pd.get_dummies(test_data_processed,columns=['MSSubClass'],dtype=int,prefix='MSSubClass')
con=(set(test_data_processed['Condition1']) | set(test_data_processed['Condition2']))
for c in con:
  if c!='Norm' :
    test_data_processed[f'Condition{c}']=((test_data_processed['Condition1']==c) | (test_data_processed['Condition2']==c)).astype(int)
test_data_processed=test_data_processed.drop(columns=['Condition1','Condition2'])
test_data_processed=pd.get_dummies(test_data_processed,columns=['LotShape'],dtype=int,prefix='LotShape')
test_data_processed=pd.get_dummies(test_data_processed,columns=['BldgType'],dtype=int,prefix='BldgType')

test_data_processed['FullBath'] = test_data_processed['FullBath'].fillna(0)
test_data_processed['HalfBath'] = test_data_processed['HalfBath'].fillna(0)
test_data_processed['BsmtHalfBath'] = test_data_processed['BsmtHalfBath'].fillna(0)
test_data_processed['BsmtFullBath'] = test_data_processed['BsmtFullBath'].fillna(0)

test_data_processed['Bathnum']=test_data_processed['FullBath']+test_data_processed['HalfBath']*0.5+test_data_processed['BsmtHalfBath']*0.5+test_data_processed['BsmtFullBath']
test_data_processed=test_data_processed.drop(columns=['BsmtFullBath','FullBath','HalfBath','BsmtHalfBath'])
test_data_processed["age"]=(test_data_processed['YrSold']-test_data_processed['YearBuilt'])
test_data_processed['IsRemodeled'] = (test_data_processed['YearBuilt'] != test_data_processed['YearRemodAdd']).astype(int)
test_data_processed['MoSold']=test_data_processed['MoSold'].astype(str)
test_data_processed=pd.get_dummies(test_data_processed,columns=['MoSold'],dtype=int,prefix='MoSold')
test_data_processed['ismodeled']=(test_data_processed['YearRemodAdd'] != test_data_processed['YearBuilt']).astype(int)

test_data_processed['TotalBsmtSF'] = test_data_processed['TotalBsmtSF'].fillna(0)
test_data_processed['TotalSF'] = test_data_processed['1stFlrSF'] + test_data_processed['2ndFlrSF'] + test_data_processed['TotalBsmtSF']
test_data_processed=test_data_processed.drop(columns=['1stFlrSF','2ndFlrSF','TotalBsmtSF'])
test_data_processed = pd.get_dummies(test_data_processed, columns=['Neighborhood'], dtype=int, prefix='Neighborhood')
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'no_access': 0, 'None': 0}
qualities=['PoolQC','GarageCond','GarageQual','FireplaceQu','KitchenQual','HeatingQC','BsmtCond','BsmtQual','ExterCond','ExterQual']
for col in qualities:
  test_data_processed[col]=test_data_processed[col].fillna('None').map(qual_map)
util={'AllPub':4,'NoSewr':3,'NoSeWa':2,'ELO':1}
test_data_processed['Utilities']=test_data_processed['Utilities'].map(util).fillna(0)
test_data_processed = pd.get_dummies(test_data_processed, columns=['LotConfig'], dtype=int, prefix='Lot')
test_data_processed['LandSlope']=test_data_processed['LandSlope'].map({'Gtl':1,'Mod':2,'Sev':3}).fillna(0)
test_data_processed = pd.get_dummies(test_data_processed, columns=['HouseStyle'],dtype=int,prefix='HouseStyle')
test_data_processed = pd.get_dummies(test_data_processed, columns=['RoofMatl'],dtype=int,prefix='RoofMatl')
test_data_processed = pd.get_dummies(test_data_processed, columns=['Foundation'],dtype=int,prefix='Foundation')
test_data_processed = pd.get_dummies(test_data_processed, columns=['RoofStyle'],dtype=int,prefix='RoofStyle')
test_data_processed = pd.get_dummies(test_data_processed, columns=['Heating'],dtype=int,prefix='Heating')
test_data_processed = pd.get_dummies(test_data_processed, columns=['Functional'],dtype=int,prefix='Functional')

con=(set(test_data_processed['Exterior1st']) | set(test_data_processed['Exterior2nd']))
for c in con:
  if c!='Norm' :
    test_data_processed[f'Exterior{c}']=((test_data_processed['Exterior1st']==c) | (test_data_processed['Exterior2nd']==c)).astype(int)
test_data_processed['MiscFeature']=test_data_processed['MiscFeature'].fillna('other')
test_data_processed = pd.get_dummies(test_data_processed, columns=['MiscFeature'],dtype=int,prefix='MiscFeature')
test_data_processed = pd.get_dummies(test_data_processed, columns=['YrSold'],dtype=int,prefix='YrSold')
test_data_processed = pd.get_dummies(test_data_processed, columns=['SaleType', 'SaleCondition'],dtype=int)
bsmt_fin_map = {
    'GLQ': 6, 'ALQ': 5, 'BLQ': 4,
    'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0
}

for col in ['BsmtFinType1', 'BsmtFinType2']:
    test_data_processed[col] = test_data_processed[col].fillna('None').map(bsmt_fin_map)
exp_map = {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}
test_data_processed['BsmtExposure'] = test_data_processed['BsmtExposure'].fillna('None').map(exp_map)
gar_map = {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0}
test_data_processed['GarageFinish'] = test_data_processed['GarageFinish'].fillna('None').map(gar_map)
test_data_processed['MasVnrArea'] = test_data_processed['MasVnrArea'].fillna(0)
test_data_processed['Electrical'] = test_data_processed['Electrical'].fillna(test_data_processed['Electrical'].mode()[0])
test_data_processed = pd.get_dummies(test_data_processed, columns=['Electrical', 'MasVnrType', 'GarageType'],dtype=int)
test_data_processed = test_data_processed.drop(columns=['Exterior1st', 'Exterior2nd'])
bsmt_cols = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF']
for col in bsmt_cols:
    test_data_processed[col] = test_data_processed[col].fillna(0)
test_data_processed['GarageYrBlt'] = test_data_processed['GarageYrBlt'].fillna(test_data_processed['YearBuilt'])
test_data_processed['GarageCars'] = test_data_processed['GarageCars'].fillna(0)
test_data_processed['GarageArea'] = test_data_processed['GarageArea'].fillna(0)
test_data_processed['Fence']=test_data_processed['Fence'].fillna('no').map({
           'GdPrv':4,
       'MnPrv'	: 3,
       'GdWo'	:2,
       'MnWw'	 :1,
       'no'	:0
})
test_data_processed['PavedDrive']=test_data_processed['PavedDrive'].map({
       'Y'	: 3,
       'P'	:2,
       'N'	 :1,

}).fillna(0)
test_data_processed['CentralAir']=test_data_processed['CentralAir'].map({
       'Y'	: 1,
       'N'	 :0

}).fillna(0)


train_cols = set(x.columns)
test_cols = set(test_data_processed.columns)

missing_in_test = list(train_cols - test_cols)
for col in missing_in_test:
    test_data_processed[col] = 0

extra_in_test = list(test_cols - train_cols)
test_data_processed = test_data_processed.drop(columns=extra_in_test)

test_data_processed = test_data_processed[x.columns]

x_test_scaled = scal.transform(test_data_processed)


In [ ]:
log_predict=model.predict(x_test_scaled)
final=np.expm1(log_predict)
subm=pd.DataFrame({'Id':id,'SalePrice':final})
subm.to_csv('submission.csv', index=False)